
All Attention implementations were referenced from the book "Build an LLM from scratch"

Code to print the attention weights was generated by AI

Both generated and copied code was edited and reviewed by Maryam Bacchus

In [71]:
import torch
import torch.nn as nn
import tiktoken

In [72]:
#simple self attention

class SimpleSelfAttention(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [97]:
tokenizer = tiktoken.get_encoding("gpt2")

# Tokenize text
text = "Hello world! The weather is lovely today"
tokens = tokenizer.encode(text)
words = text.split()

d_in = 3
d_out = 2

# Create embeddings from tokens
torch.manual_seed(123)
inputs = torch.randn(len(tokens), d_in)

torch.manual_seed(123)
simple_sa = SimpleSelfAttention(d_in, d_out)
output = simple_sa(inputs)

print("Text: ", text)
print("Words: ", words)

# Extract and print attention weights
with torch.no_grad():
    keys = inputs @ simple_sa.W_key
    queries = inputs @ simple_sa.W_query
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(attn_scores, dim=-1).numpy()

print("\nAttention weights:")
for word in words:
    print(f"{word:8}", end='')
print()
for i, word in enumerate(words):
    print(f"{word:8} [", end='')
    for j in range(len(words)):
        print(f"{attn_weights[i][j]:.2f}", end='')
        if j < len(words) - 1:
            print(", ", end='')
    print("]")

Text:  Hello world! The weather is lovely today
Words:  ['Hello', 'world!', 'The', 'weather', 'is', 'lovely', 'today']

Attention weights:
Hello   world!  The     weather is      lovely  today   
Hello    [0.13, 0.11, 0.14, 0.15, 0.10, 0.17, 0.10]
world!   [0.09, 0.15, 0.09, 0.07, 0.17, 0.05, 0.20]
The      [0.13, 0.08, 0.14, 0.18, 0.06, 0.30, 0.05]
weather  [0.09, 0.03, 0.10, 0.17, 0.02, 0.57, 0.01]
is       [0.01, 0.10, 0.01, 0.00, 0.27, 0.00, 0.35]
lovely   [0.01, 0.00, 0.02, 0.06, 0.00, 0.90, 0.00]
today    [0.04, 0.14, 0.04, 0.02, 0.23, 0.01, 0.28]


In [74]:
# Scaled Dot-Product Attention

class ScaledDotProduct(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [96]:
tokenizer = tiktoken.get_encoding("gpt2")

# Tokenize text
text = "The cat sat on the mat"
tokens = tokenizer.encode(text)
words = text.split()

d_in = 3
d_out = 2

# Create embeddings from tokens
torch.manual_seed(789)
inputs = torch.randn(len(tokens), d_in)

torch.manual_seed(789)
sa_v2 = ScaledDotProduct(d_in, d_out)
output = sa_v2(inputs)

print("Text: ", text)
print("Words: ", words)

# Extract and print attention weights
with torch.no_grad():
    keys = sa_v2.W_key(inputs)
    queries = sa_v2.W_query(inputs)
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1).numpy()

print("\nAttention weights:")
for word in words:
    print(f"{word:8}", end='')
print()
for i, word in enumerate(words):
    print(f"{word:8} [", end='')
    for j in range(len(words)):
        print(f"{attn_weights[i][j]:.2f}", end='')
        if j < len(words) - 1:
            print(", ", end='')
    print("]")

Text:  The cat sat on the mat
Words:  ['The', 'cat', 'sat', 'on', 'the', 'mat']

Attention weights:
The     cat     sat     on      the     mat     
The      [0.21, 0.11, 0.18, 0.13, 0.26, 0.10]
cat      [0.20, 0.16, 0.19, 0.14, 0.17, 0.14]
sat      [0.17, 0.15, 0.16, 0.17, 0.20, 0.15]
on       [0.12, 0.22, 0.14, 0.19, 0.09, 0.24]
the      [0.17, 0.16, 0.17, 0.17, 0.18, 0.16]
mat      [0.16, 0.20, 0.17, 0.16, 0.12, 0.19]


In [76]:
# Trainable attention

class TrainableAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


In [98]:
tokenizer = tiktoken.get_encoding("gpt2")

# Tokenize text
text = "The cat sat on the mat"
tokens = tokenizer.encode(text)
words = text.split()

batch_size = 2
context_length = len(tokens)
d_in = 3
d_out = 2

# Create batch with embeddings from tokens
torch.manual_seed(123)
batch = torch.randn(batch_size, context_length, d_in)

torch.manual_seed(123)
ca = TrainableAttention(d_in, d_out, context_length, dropout=0.0)

context_vecs = ca(batch)

print("Text: ", text)
print("Words: ", words)

# Extract and print attention weights
with torch.no_grad():
    keys = ca.W_key(batch)
    queries = ca.W_query(batch)
    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(ca.mask.bool()[:context_length, :context_length], -torch.inf)
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)[0].numpy()

print("\nAttention weights:")
for word in words:
    print(f"{word:8}", end='')
print()
for i, word in enumerate(words):
    print(f"{word:8} [", end='')
    for j in range(len(words)):
        if j <= i:
            print(f"{attn_weights[i][j]:.2f}", end='')
        else:
            print("--", end='')
        if j < len(words) - 1:
            print(", ", end='')
    print("]")

Text:  The cat sat on the mat
Words:  ['The', 'cat', 'sat', 'on', 'the', 'mat']

Attention weights:
The     cat     sat     on      the     mat     
The      [1.00, --, --, --, --, --]
cat      [0.50, 0.50, --, --, --, --]
sat      [0.33, 0.32, 0.35, --, --, --]
on       [0.18, 0.18, 0.15, 0.49, --, --]
the      [0.17, 0.19, 0.16, 0.24, 0.23, --]
mat      [0.16, 0.16, 0.15, 0.21, 0.14, 0.17]


In [78]:
# Multihead attention

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


In [100]:
tokenizer = tiktoken.get_encoding("gpt2")

# Tokenize text
text = "I enjoy long walks along the seashore"
tokens = tokenizer.encode(text)
words = text.split()

batch_size = 2
context_length = len(tokens)
d_in = 3
d_out = 8
num_heads = [1, 2, 4]

torch.manual_seed(400)
batch = torch.randn(batch_size, context_length, d_in)

print("Text: ", text)
print("Words: ", words)
print("\n" + "="*70 + "\n")

for head in num_heads:
    torch.manual_seed(400)
    mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=head)
    context_vecs = mha(batch)
    print("Head dimension: ", d_out // head)

    # Extract and print attention weights
    with torch.no_grad():
        b, num_tokens, d = batch.shape
        keys = mha.W_key(batch)
        queries = mha.W_query(batch)

        keys = keys.view(b, num_tokens, head, mha.head_dim)
        queries = queries.view(b, num_tokens, head, mha.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = mha.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    # Print each head
    for head_idx in range(head):
        print(f"\nHead {head_idx + 1} attention weights:")
        for word in words:
            print(f"{word:8}", end='')
        print()

        weights = attn_weights[0, head_idx].numpy()
        for i, word in enumerate(words):
            print(f"{word:8} [", end='')
            for j in range(len(words)):
                if j <= i:
                    print(f"{weights[i][j]:.2f}", end='')
                else:
                    print("--", end='')
                if j < len(words) - 1:
                    print(", ", end='')
            print("]")

    print("\n" + "="*70 + "\n")

Text:  I enjoy long walks along the seashore
Words:  ['I', 'enjoy', 'long', 'walks', 'along', 'the', 'seashore']


Head dimension:  8

Head 1 attention weights:
I       enjoy   long    walks   along   the     seashore
I        [1.00, --, --, --, --, --, --]
enjoy    [0.62, 0.38, --, --, --, --, --]
long     [0.31, 0.31, 0.38, --, --, --, --]
walks    [0.24, 0.29, 0.23, 0.24, --, --, --]
along    [0.15, 0.11, 0.23, 0.15, 0.36, --, --]
the      [0.19, 0.18, 0.11, 0.17, 0.07, 0.28, --]
seashore [0.14, 0.10, 0.06, 0.11, 0.03, 0.21, 0.35]


Head dimension:  4

Head 1 attention weights:
I       enjoy   long    walks   along   the     seashore
I        [1.00, --, --, --, --, --, --]
enjoy    [0.61, 0.39, --, --, --, --, --]
long     [0.31, 0.32, 0.37, --, --, --, --]
walks    [0.23, 0.23, 0.25, 0.28, --, --, --]
along    [0.16, 0.17, 0.20, 0.13, 0.35, --, --]
the      [0.20, 0.13, 0.11, 0.21, 0.05, 0.30, --]
seashore [0.14, 0.08, 0.05, 0.12, 0.02, 0.22, 0.38]

Head 2 attention weights:
I     

##EXERCISES

In [86]:
# Comparing single head vs multi attention head

print("-"*70)

# Tokenizer
tokenizer = tiktoken.get_encoding("gpt2")
text = "Your journey starts with one step"
tokens = tokenizer.encode(text)
words = text.split()

print("\nText: ",  text)
print("Words: ", words)
print("Tokens: ",tokens)
print("\n")

d_in = 3
d_out = 8
num_tokens = len(tokens)
torch.manual_seed(42)
sample_input = torch.randn(1, num_tokens, d_in)

print("-"*70)
# Single head attention

print("\nSingle head attention\n")
torch.manual_seed(456)
single_head = TrainableAttention(d_in, d_out, num_tokens, dropout=0.0)
output_single = single_head(sample_input)

# Get Attention weights
with torch.no_grad():
    keys = single_head.W_key(sample_input)
    queries = single_head.W_query(sample_input)
    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(single_head.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
    single_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

# Print attention matrix
weights = single_weights[0].numpy()
print("\nAttention weights:")
print("         ", end='')
for word in words:
    print(f"{word:10}", end='')
print()

for i, word in enumerate(words):
    print(f"{word:8} [", end='')
    for j in range(len(words)):
        if j <= i:
            print(f"{weights[i][j]:.2f}", end='')
        else:
            print("--", end='')
        if j < len(words) - 1:
            print(", ", end='')
    print("]")

print("\n")
print("-"*70)
# Multi head attention
print("\nMulti head attention\n")

torch.manual_seed(456)
num_heads = 2
multi_head = MultiHeadAttention(d_in, d_out, num_tokens, dropout=0.0, num_heads=num_heads)
output_multi = multi_head(sample_input)

# Get attention weights
with torch.no_grad():
    b, num_tok, d = sample_input.shape
    keys = multi_head.W_key(sample_input)
    queries = multi_head.W_query(sample_input)

    keys = keys.view(b, num_tok, num_heads, multi_head.head_dim)
    queries = queries.view(b, num_tok, num_heads, multi_head.head_dim)

    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)

    attn_scores = queries @ keys.transpose(2, 3)
    mask_bool = multi_head.mask.bool()[:num_tok, :num_tok]
    attn_scores.masked_fill_(mask_bool, -torch.inf)
    multi_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)


# Print attention for each head
for head_idx in range(num_heads):
    print(f"\nHead {head_idx + 1} attention weights:")
    print("         ", end='')
    for word in words:
        print(f"{word:10}", end='')
    print()

    weights = multi_weights[0, head_idx].numpy()
    for i, word in enumerate(words):
        print(f"{word:8} [", end='')
        for j in range(len(words)):
            if j <= i:
                print(f"{weights[i][j]:.2f}", end='')
            else:
                print("--", end='')
            if j < len(words) - 1:
                print(", ", end='')
        print("]")


----------------------------------------------------------------------

Text:  Your journey starts with one step
Words:  ['Your', 'journey', 'starts', 'with', 'one', 'step']
Tokens:  [7120, 7002, 4940, 351, 530, 2239]


----------------------------------------------------------------------

Single head attention


Attention weights:
         Your      journey   starts    with      one       step      
Your     [1.00, --, --, --, --, --]
journey  [0.50, 0.50, --, --, --, --]
starts   [0.37, 0.36, 0.27, --, --, --]
with     [0.40, 0.25, 0.22, 0.13, --, --]
one      [0.20, 0.20, 0.20, 0.19, 0.20, --]
step     [0.12, 0.16, 0.17, 0.24, 0.18, 0.13]


----------------------------------------------------------------------

Multi head attention


Head 1 attention weights:
         Your      journey   starts    with      one       step      
Your     [1.00, --, --, --, --, --]
journey  [0.50, 0.50, --, --, --, --]
starts   [0.49, 0.24, 0.27, --, --, --]
with     [0.45, 0.25, 0.20, 0.10, --, --]


In [89]:
tokenizer = tiktoken.get_encoding("gpt2")

# Narrative
narrative_text = "The cat sat on the mat"
narrative_tokens = tokenizer.encode(narrative_text)
narrative_words = narrative_text.split()
print("\n Narrative: ", narrative_text)
print("\n Tokens: ", narrative_tokens)


torch.manual_seed(42)
embedding_dim = 3
narrative_embeddings = torch.randn(len(narrative_tokens), embedding_dim)
narrative_input = narrative_embeddings.unsqueeze(0)
attn_narrative = TrainableAttention(d_in=3, d_out=8, context_length=len(narrative_tokens), dropout=0.0)
output_narrative = attn_narrative(narrative_input)
print("\n Output shape: ", output_narrative.shape)

# Extract and print attention weights
with torch.no_grad():
    keys = attn_narrative.W_key(narrative_input)
    queries = attn_narrative.W_query(narrative_input)
    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(attn_narrative.mask.bool()[:len(narrative_tokens), :len(narrative_tokens)], -torch.inf)
    weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)[0].numpy()

print("\n Attention weights:\n")
print("         ", end='')
for word in narrative_words:
    print(f"{word:8}", end='')
print()
for i, word in enumerate(narrative_words):
    print(f"{word:8} [", end='')
    for j in range(len(narrative_words)):
        if j <= i:
            print(f"{weights[i][j]:.2f}", end='')
        else:
            print("--", end='')
        if j < len(narrative_words) - 1:
            print(", ", end='')
    print("]")

print("\n")
print("-"*70)

# 2. Code
code_text = "def add(a, b): return a + b"
code_tokens = tokenizer.encode(code_text)
code_words = code_text.split()
print("\n Code: ", code_text)
print("\n Tokens: ", code_tokens)

torch.manual_seed(42)
code_embeddings = torch.randn(len(code_tokens), embedding_dim)
code_input = code_embeddings.unsqueeze(0)
attn_code = TrainableAttention(d_in=3, d_out=8, context_length=len(code_tokens), dropout=0.0)
output_code = attn_code(code_input)
print("\n Output shape: ", output_code.shape)

# Extract and print attention weights
with torch.no_grad():
    keys = attn_code.W_key(code_input)
    queries = attn_code.W_query(code_input)
    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(attn_code.mask.bool()[:len(code_tokens), :len(code_tokens)], -torch.inf)
    weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)[0].numpy()

print("\n Attention weights:\n")
print("         ", end='')
for word in code_words:
    print(f"{word:8}", end='')
print()
for i, word in enumerate(code_words):
    print(f"{word:8} [", end='')
    for j in range(len(code_words)):
        if j <= i:
            print(f"{weights[i][j]:.2f}", end='')
        else:
            print("--", end='')
        if j < len(code_words) - 1:
            print(", ", end='')
    print("]")

print("\n")
print("-"*70)

# 3. Poertry
poetry_text = "Roses are red, violets are blue"
poetry_tokens = tokenizer.encode(poetry_text)
poetry_words = poetry_text.replace(',', '').split()
print("\n Poetry: ", poetry_text)
print("\n Tokens: ", poetry_tokens)

torch.manual_seed(42)
poetry_embeddings = torch.randn(len(poetry_tokens), embedding_dim)
poetry_input = poetry_embeddings.unsqueeze(0)
attn_poetry = TrainableAttention(d_in=3, d_out=8, context_length=len(poetry_tokens), dropout=0.0)
output_poetry = attn_poetry(poetry_input)
print("\n Output shape: ", output_poetry.shape)

# Extract and print attention weights
with torch.no_grad():
    keys = attn_poetry.W_key(poetry_input)
    queries = attn_poetry.W_query(poetry_input)
    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(attn_poetry.mask.bool()[:len(poetry_tokens), :len(poetry_tokens)], -torch.inf)
    weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)[0].numpy()

print("\n Attention weights:\n")
print("         ", end='')
for word in poetry_words:
    print(f"{word:10}", end='')
print()
for i, word in enumerate(poetry_words):
    print(f"{word:10} [", end='')
    for j in range(len(poetry_words)):
        if j <= i:
            print(f"{weights[i][j]:.2f}", end='')
        else:
            print("--", end='')
        if j < len(poetry_words) - 1:
            print(", ", end='')
    print("]")


 Narrative:  The cat sat on the mat

 Tokens:  [464, 3797, 3332, 319, 262, 2603]

 Output shape:  torch.Size([1, 6, 8])

 Attention weights:

         The     cat     sat     on      the     mat     
The      [1.00, --, --, --, --, --]
cat      [0.51, 0.49, --, --, --, --]
sat      [0.40, 0.25, 0.35, --, --, --]
on       [0.26, 0.23, 0.27, 0.24, --, --]
the      [0.19, 0.20, 0.19, 0.20, 0.22, --]
mat      [0.17, 0.16, 0.16, 0.15, 0.18, 0.18]


----------------------------------------------------------------------

 Code:  def add(a, b): return a + b

 Tokens:  [4299, 751, 7, 64, 11, 275, 2599, 1441, 257, 1343, 275]

 Output shape:  torch.Size([1, 11, 8])

 Attention weights:

         def     add(a,  b):     return  a       +       b       
def      [1.00, --, --, --, --, --, --]
add(a,   [0.17, 0.83, --, --, --, --, --]
b):      [0.24, 0.31, 0.45, --, --, --, --]
return   [0.28, 0.07, 0.28, 0.37, --, --, --]
a        [0.14, 0.27, 0.18, 0.21, 0.20, --, --]
+        [0.25, 0.10, 0.12, 